# CNN / ResNet training and explainability
This notebook is split from the original thesis workflow to keep each stage focused and easier to maintain.


In [ ]:
# NOTE: comment translated/cleaned for English-only repository.
from pathlib import Path
import os
_here = Path.cwd().resolve()
if _here.name == "notebooks":
    os.chdir(_here.parent)
elif not (_here / "image").is_dir() and (_here.parent / "image").is_dir():
    os.chdir(_here.parent)
print("Project root:", Path.cwd())


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from collections import Counter

error_counter = Counter()

# NOTE: comment translated/cleaned for English-only repository.
save_dir = r"C:\Users\Adolph\Desktop\graduation thesis\image"
os.makedirs(save_dir, exist_ok=True)
num_samples = 4000

# NOTE: comment translated/cleaned for English-only repository.
df = df.copy()
df['is_valuable'] = (df['window_score'] > 0.5).astype(int)

# NOTE: comment translated/cleaned for English-only repository.
df_positive = df[df['is_valuable'] == 1]
df_negative = df[df['is_valuable'] == 0].sample(n=max(0, num_samples - len(df_positive)), random_state=42)
df_selected = pd.concat([df_positive, df_negative]).sample(frac=1.0, random_state=42).reset_index(drop=True)

# NOTE: comment translated/cleaned for English-only repository.
def draw_pass_frame_as_image(df_players, pass_row, save_path):
    plt.figure(figsize=(10, 6.5))
    ax = plt.gca()
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 68)
    ax.set_facecolor('#99ff99')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

# NOTE: comment translated/cleaned for English-only repository.
    passer = df_players[df_players['is_passer']]
    x0, y0 = passer['posX'].values[0], passer['posY'].values[0]
    x1 = pass_row.get('receivedPlayerPosX', pass_row.get('receivedPlayerPosXh', -1))
    y1 = pass_row.get('receivedPlayerPosY', pass_row.get('receivedPlayerPosYh', -1))

# NOTE: comment translated/cleaned for English-only repository.
    ax.scatter(x0, y0, color='blue', s=100, label='Passer')
    ax.scatter(df_players[df_players['is_teammate']]['posX'],
               df_players[df_players['is_teammate']]['posY'],
               color='lightgray', s=60, label='Teammates')
    ax.scatter(df_players[df_players['is_opponent']]['posX'],
               df_players[df_players['is_opponent']]['posY'],
               color='black', s=60, label='Opponents')

    if x1 > 0 and y1 > 0:
        ax.scatter(x1, y1, color='gold', edgecolors='black', s=100, label='Receiver', zorder=5)

# NOTE: comment translated/cleaned for English-only repository.
        ax.arrow(x0, y0, x1 - x0, y1 - y0, width=0.3, head_width=2.2,
                 length_includes_head=True, color='red', alpha=0.9)

# NOTE: comment translated/cleaned for English-only repository.
        dx, dy = x1 - x0, y1 - y0
        norm = np.sqrt(dx**2 + dy**2)
        if norm > 0:
            dx, dy = dx / norm, dy / norm
            normal = np.array([-dy, dx])
            opps = df_players[df_players['is_opponent']]
            for _, row in opps.iterrows():
                ox, oy = row['posX'], row['posY']
                vec = np.array([ox - x0, oy - y0])
                proj = np.dot(vec, [dx, dy])
                dist = abs(np.dot(vec, normal))
                if 0 <= proj <= norm and dist <= 2:
                    ax.scatter(ox, oy, s=160, facecolors='none', edgecolors='purple',
                               linewidths=2, label='In Path' if 'In Path' not in ax.get_legend_handles_labels()[1] else "")

# NOTE: comment translated/cleaned for English-only repository.
        nearest_vec = None
        min_dist = float('inf')
        for _, row in opps.iterrows():
            dxo = row['posX'] - x0
            dyo = row['posY'] - y0
            dist = np.hypot(dxo, dyo)
            if dist > 0 and dist < min_dist:
                min_dist = dist
                nearest_vec = np.array([dxo, dyo]) / dist
        if nearest_vec is not None:
            ax.arrow(x0, y0, nearest_vec[0]*12, nearest_vec[1]*12,
                     width=0.2, head_width=1.2, color='blue', alpha=0.6,
                     label='Angle to Nearest Opponent' if 'Angle to Nearest Opponent' not in ax.get_legend_handles_labels()[1] else "")

# NOTE: comment translated/cleaned for English-only repository.
    pressure = pass_row.get('pressure_level_num', 0)
    color = {0: 'grey', 1: '#a0c4ff', 2: 'red'}.get(pressure, 'gray')
    circle = Circle((x0, y0), radius=6, color=color, alpha=0.15, label=f'Pressure Level {pressure}')
    ax.add_patch(circle)

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
    plt.close()

# NOTE: comment translated/cleaned for English-only repository.
label_rows = []
count = 0

for idx, pass_row in df_selected.iterrows():
    ts = pass_row['timestamp']
    if pd.isna(ts):
        error_counter["timestamp NaN"] += 1
        continue
    try:
        pass_row_out, df_players = extract_frame_players_and_roles(df_coordinates, pass_row)

        img_name = f"pass_{idx:04d}.png"
        img_path = os.path.join(save_dir, img_name)
        draw_pass_frame_as_image(df_players, pass_row_out, img_path)

        label_rows.append({
            'filename': img_name,
            'is_valuable': int(pass_row['is_valuable'])
        })
        count += 1

    except Exception as e:
        error_counter[str(e)] += 1
        continue

# NOTE: comment translated/cleaned for English-only repository.
df_labels = pd.DataFrame(label_rows)
csv_path = os.path.join(save_dir, "pass_labels_valuable.csv")
df_labels.to_csv(csv_path, index=False)

print(f"Images generated: {count}")
print(f"Label CSV saved to: {csv_path}")
print(df_labels['is_valuable'].value_counts())
print("Error Summary:")
for err, cnt in error_counter.most_common():
    print(f"{err}: {cnt}")


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# NOTE: comment translated/cleaned for English-only repository.
image_dir = r"C:\Users\Adolph\Desktop\graduation thesis\image"
label_path = os.path.join(image_dir, "pass_labels_valuable.csv")
df = pd.read_csv(label_path)

# NOTE: comment translated/cleaned for English-only repository.
df = df.dropna(subset=['is_valuable'])
df['is_valuable'] = df['is_valuable'].astype(int)

# NOTE: comment translated/cleaned for English-only repository.
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_valuable'])

# NOTE: comment translated/cleaned for English-only repository.
class ImageDataset(Sequence):
    def __init__(self, df, batch_size=32, image_size=(224, 224), shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.image_size = image_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        images, labels = [], []
        for _, row in batch_df.iterrows():
            img_path = os.path.join(image_dir, row['filename'])
            img = load_img(img_path, target_size=self.image_size)
            img_array = preprocess_input(img_to_array(img))  #    ResNet50    
            images.append(img_array)
            labels.append(row['is_valuable'])
        return np.array(images), np.array(labels, dtype=np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# NOTE: comment translated/cleaned for English-only repository.
train_gen = ImageDataset(train_df)
val_gen = ImageDataset(val_df, shuffle=False)

# NOTE: comment translated/cleaned for English-only repository.
def build_resnet_model(input_shape=(224, 224, 3)):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  #       
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs=base_model.input, outputs=output)

model = build_resnet_model()
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss=losses.BinaryCrossentropy(),
    metrics=[
        metrics.BinaryAccuracy(),
        metrics.Precision(),
        metrics.Recall(),
        metrics.AUC()
    ]
)

# NOTE: comment translated/cleaned for English-only repository.
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint("best_resnet_model.h5", save_best_only=True)
]

model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight={0: 1.0, 1: 3.0},
    callbacks=callbacks,
    verbose=1
)

# NOTE: comment translated/cleaned for English-only repository.
model.save("resnet50_cnn_is_valuable_final.h5")


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# NOTE: comment translated/cleaned for English-only repository.
image_dir = r"C:\Users\Adolph\Desktop\graduation thesis\image"
label_path = os.path.join(image_dir, "pass_labels_valuable.csv")
df = pd.read_csv(label_path)

# NOTE: comment translated/cleaned for English-only repository.
df = df.dropna(subset=['is_valuable'])
df['is_valuable'] = df['is_valuable'].astype(int)

# NOTE: comment translated/cleaned for English-only repository.
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_valuable'])

# NOTE: comment translated/cleaned for English-only repository.
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# NOTE: comment translated/cleaned for English-only repository.
class ImageDataset(Sequence):
    def __init__(self, df, batch_size=32, image_size=(224, 224), shuffle=True, augment=False):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.image_size = image_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        images, labels = [], []
        for _, row in batch_df.iterrows():
            img_path = os.path.join(image_dir, row['filename'])
            img = load_img(img_path, target_size=self.image_size)
            img_array = img_to_array(img)
            if self.augment:
                img_array = datagen.random_transform(img_array)
            img_array = preprocess_input(img_array)
            images.append(img_array)
            labels.append(row['is_valuable'])
        return np.array(images), np.array(labels, dtype=np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# NOTE: comment translated/cleaned for English-only repository.
train_gen = ImageDataset(train_df, augment=True)
val_gen = ImageDataset(val_df, shuffle=False)

# NOTE: comment translated/cleaned for English-only repository.
def build_resnet_model(input_shape=(224, 224, 3)):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = True

    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs=base_model.input, outputs=output)

# NOTE: comment translated/cleaned for English-only repository.
def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        return -tf.reduce_mean(alpha * tf.pow(1. - pt, gamma) * tf.math.log(pt))
    return loss

# NOTE: comment translated/cleaned for English-only repository.
model = build_resnet_model()
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),  #        
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=[
        metrics.BinaryAccuracy(name='binary_accuracy'),
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall'),
        metrics.AUC(name='auc')
    ]
)

# NOTE: comment translated/cleaned for English-only repository.
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint("best_resnet_model_focal_finetune.h5", save_best_only=True)
]

# NOTE: comment translated/cleaned for English-only repository.
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight={0: 1.0, 1: 3.0},  #      balanced       
    callbacks=callbacks,
    verbose=1
)

# NOTE: comment translated/cleaned for English-only repository.
model.save("resnet50_focal_finetuned_final.h5")


In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input

# NOTE: comment translated/cleaned for English-only repository.
image_dir = r"C:\Users\Adolph\Desktop\graduation thesis\cnn_pass_images"
model_path = "resnet50_focal_finetuned_final.h5"
image_filename = "cnn_ready_pass_1.png"  #         
last_conv_layer_name = "conv5_block3_out"  # ResNet50        

# NOTE: comment translated/cleaned for English-only repository.
model = tf.keras.models.load_model(model_path, compile=False)

# NOTE: comment translated/cleaned for English-only repository.
def load_and_preprocess_image(img_path, target_size=(224, 224)):
    img = load_img(img_path, target_size=target_size)
    raw_img = img.copy()
    img_array = img_to_array(img)
    img_array = preprocess_input(img_array)
    return np.expand_dims(img_array, axis=0), raw_img

img_path = os.path.join(image_dir, image_filename)
img_tensor, raw_img = load_and_preprocess_image(img_path)

# NOTE: comment translated/cleaned for English-only repository.
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap + tf.keras.backend.epsilon())
    return heatmap.numpy()

# NOTE: comment translated/cleaned for English-only repository.
def display_gradcam(heatmap, original_img, alpha=0.5, cmap='jet'):
    heatmap = np.uint8(255 * heatmap)
    heatmap_resized = cv2.resize(heatmap, (original_img.size[0], original_img.size[1]))
    heatmap_colored = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

    img = np.array(original_img)
    if img.shape[-1] == 4:
        img = img[..., :3]

# NOTE: comment translated/cleaned for English-only repository.
    heatmap_colored = cv2.convertScaleAbs(heatmap_colored, alpha=1.2, beta=30)

    superimposed_img = cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(superimposed_img)
    plt.axis('off')
    plt.title("Grad-CAM: Model Attention on Valuable Pass")
    plt.show()


# NOTE: comment translated/cleaned for English-only repository.
heatmap = make_gradcam_heatmap(img_tensor, model, last_conv_layer_name)
display_gradcam(heatmap, raw_img)


In [ ]:
pip install opencv-python-headless


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrow

# NOTE: comment translated/cleaned for English-only repository.
def extract_frame_players_and_roles(coordinates_df, row):
    timestamp = row['timestamp']
    passer_id = row['passedPlayerId']
    team_id = row['passedPlayerTeamId']

    frame = coordinates_df[coordinates_df['timestamp'] == timestamp]
    if frame.empty or pd.isna(timestamp):
        raise ValueError(f"No coordinate data found for timestamp {timestamp}")
    frame = frame.iloc[0]

    player_rows = []
    for i in range(40):
        pid = frame.get(f'playerId{i}', -100)
        x = frame.get(f'posX{i}', -100)
        y = frame.get(f'posY{i}', -100)
        if pid == -100 or x == -100 or y == -100:
            continue
        player_rows.append({'player_index': i, 'player_id': pid, 'posX': x, 'posY': y})

    df_players = pd.DataFrame(player_rows)

    teammate_ids = [row.get(f'teamMatePlayerId{i}') for i in range(1, 11)]
    opponent_ids = [row.get(f'opponentPlayerId{i}') for i in range(1, 12)]

    df_players['is_passer'] = df_players['player_id'] == passer_id
    df_players['is_teammate'] = df_players['player_id'].isin(teammate_ids)
    df_players['is_opponent'] = df_players['player_id'].isin(opponent_ids)

    return row, df_players

# NOTE: comment translated/cleaned for English-only repository.
def draw_cnn_ready_pass_scene(df_players, pass_row, save_path=None):
    fig, ax = plt.subplots(figsize=(10, 6.5))  #       
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 68)
    ax.set_facecolor('#60B860')  #     
    ax.axis('off')

# NOTE: comment translated/cleaned for English-only repository.
    passer = df_players[df_players['is_passer']]
    x0, y0 = passer['posX'].values[0], passer['posY'].values[0]
    ax.scatter(x0, y0, color='blue', s=80, zorder=3)

# NOTE: comment translated/cleaned for English-only repository.
    if pass_row.get('isSucceeded') == 0:
        x1 = pass_row.get('receivedPlayerPosXh', -1)
        y1 = pass_row.get('receivedPlayerPosYh', -1)
    else:
        x1 = pass_row.get('receivedPlayerPosX', -1)
        y1 = pass_row.get('receivedPlayerPosY', -1)

    if x1 > 0 and y1 > 0:
        ax.scatter(x1, y1, color='yellow', s=80, edgecolors='black', zorder=3)
        ax.add_patch(FancyArrow(x0, y0, x1 - x0, y1 - y0,
                                width=0.3, head_width=2.0, head_length=2.5,
                                length_includes_head=True, color='red', zorder=2))

# NOTE: comment translated/cleaned for English-only repository.
    ax.scatter(df_players[df_players['is_teammate']]['posX'],
               df_players[df_players['is_teammate']]['posY'],
               color='white', s=60, zorder=1)

# NOTE: comment translated/cleaned for English-only repository.
    ax.scatter(df_players[df_players['is_opponent']]['posX'],
               df_players[df_players['is_opponent']]['posY'],
               color='black', s=60, zorder=1)

# NOTE: comment translated/cleaned for English-only repository.
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight', pad_inches=0, facecolor=ax.get_facecolor())  #     dpi   100
        plt.close()
    else:
        plt.tight_layout()
        plt.show()

# NOTE: comment translated/cleaned for English-only repository.
image_save_dir = r"C:\Users\Adolph\Desktop\graduation thesis\cnn_pass_images"
os.makedirs(image_save_dir, exist_ok=True)

for i in range(5):
    row = df_passes.iloc[i]
    try:
        pass_row, frame_players = extract_frame_players_and_roles(df_coordinates, row)
        save_path = os.path.join(image_save_dir, f"cnn_ready_pass_{i}.png")
        draw_cnn_ready_pass_scene(frame_players, pass_row, save_path=save_path)
        print(f"  Saved: {save_path}")
    except Exception as e:
        print(f"   Skipped {i} due to error: {e}")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# NOTE: comment translated/cleaned for English-only repository.
model_path = r"C:\Users\Adolph\graduation thesis\resnet50_cnn_is_valuable_final.h5"
model = load_model(model_path)
last_conv_layer_name = "conv5_block3_out"  # ResNet50         

# NOTE: comment translated/cleaned for English-only repository.
def compute_gradcam(model, image_tensor, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_tensor)
        pred_index = tf.argmax(predictions[0])
        class_output = predictions[:, pred_index]

    grads = tape.gradient(class_output, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + tf.keras.backend.epsilon()
    return heatmap.numpy()

# NOTE: comment translated/cleaned for English-only repository.
def overlay_gradcam_on_image(img_path, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET, output_path=None):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_resized = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_resized, colormap)

    superimposed_img = cv2.addWeighted(img_rgb, 1 - alpha, heatmap_color, alpha, 0)

    plt.figure(figsize=(6, 4))
    plt.imshow(superimposed_img)
    plt.title("Grad-CAM: Model Attention on Valuable Pass", fontsize=14)
    plt.axis('off')
    if output_path:
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
        plt.close()
    else:
        plt.show()

# NOTE: comment translated/cleaned for English-only repository.
img_dir = r"C:\Users\Adolph\Desktop\graduation thesis\cnn_pass_images"
save_dir = os.path.join(img_dir, "gradcam_output")
os.makedirs(save_dir, exist_ok=True)

image_files = [f for f in os.listdir(img_dir) if f.endswith('.png') and f.startswith("cnn_ready")]
print(f"Found {len(image_files)} images to process.")

for fname in image_files:
    try:
        path = os.path.join(img_dir, fname)
        print(f"\U0001F50D Processing: {path}")

# NOTE: comment translated/cleaned for English-only repository.
        img = load_img(path, target_size=(224, 224))
        img_array = img_to_array(img)
        img_tensor = np.expand_dims(img_array, axis=0)
        img_tensor = tf.keras.applications.resnet50.preprocess_input(img_tensor)

# NOTE: comment translated/cleaned for English-only repository.
        heatmap = compute_gradcam(model, img_tensor, last_conv_layer_name)

# NOTE: comment translated/cleaned for English-only repository.
        save_path = os.path.join(save_dir, f"gradcam_{fname}")
        overlay_gradcam_on_image(path, heatmap, output_path=save_path)
        print(f"  Saved to {save_path}")

    except Exception as e:
        print(f"  Failed on {fname}: {e}")


In [ ]:
print(model.output)


In [ ]:


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from collections import Counter

error_counter = Counter()

# NOTE: comment translated/cleaned for English-only repository.
save_dir = r"C:\Users\Adolph\Desktop\graduation thesis\predict_image"
os.makedirs(save_dir, exist_ok=True)
num_samples = 4000

# NOTE: comment translated/cleaned for English-only repository.
df = df.copy()
df['is_valuable'] = (df['window_score'] > 0.5).astype(int)

# NOTE: comment translated/cleaned for English-only repository.
df_positive = df[df['is_valuable'] == 1]
df_negative = df[df['is_valuable'] == 0].sample(n=max(0, num_samples - len(df_positive)), random_state=42)
df_selected = pd.concat([df_positive, df_negative]).sample(frac=1.0, random_state=42).reset_index(drop=True)

# NOTE: comment translated/cleaned for English-only repository.
def draw_pass_frame_as_image(df_players, pass_row, save_path):
    plt.figure(figsize=(10, 6.5))
    ax = plt.gca()
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 68)
    ax.set_facecolor('#99ff99')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

# NOTE: comment translated/cleaned for English-only repository.
    passer = df_players[df_players['is_passer']]
    x0, y0 = passer['posX'].values[0], passer['posY'].values[0]
    x1 = pass_row.get('receivedPlayerPosX', pass_row.get('receivedPlayerPosXh', -1))
    y1 = pass_row.get('receivedPlayerPosY', pass_row.get('receivedPlayerPosYh', -1))

# NOTE: comment translated/cleaned for English-only repository.
    ax.scatter(x0, y0, color='blue', s=100, label='Passer')
    ax.scatter(df_players[df_players['is_teammate']]['posX'],
               df_players[df_players['is_teammate']]['posY'],
               color='lightgray', s=60, label='Teammates')
    ax.scatter(df_players[df_players['is_opponent']]['posX'],
               df_players[df_players['is_opponent']]['posY'],
               color='black', s=60, label='Opponents')

    if x1 > 0 and y1 > 0:
        ax.scatter(x1, y1, color='gold', edgecolors='black', s=100, label='Receiver', zorder=5)

# NOTE: comment translated/cleaned for English-only repository.
        ax.arrow(x0, y0, x1 - x0, y1 - y0, width=0.3, head_width=2.2,
                 length_includes_head=True, color='red', alpha=0.9)

# NOTE: comment translated/cleaned for English-only repository.
        dx, dy = x1 - x0, y1 - y0
        norm = np.sqrt(dx**2 + dy**2)
        if norm > 0:
            dx, dy = dx / norm, dy / norm
            normal = np.array([-dy, dx])
            opps = df_players[df_players['is_opponent']]
            for _, row in opps.iterrows():
                ox, oy = row['posX'], row['posY']
                vec = np.array([ox - x0, oy - y0])
                proj = np.dot(vec, [dx, dy])
                dist = abs(np.dot(vec, normal))
                if 0 <= proj <= norm and dist <= 2:
                    ax.scatter(ox, oy, s=160, facecolors='none', edgecolors='purple',
                               linewidths=2, label='In Path' if 'In Path' not in ax.get_legend_handles_labels()[1] else "")

# NOTE: comment translated/cleaned for English-only repository.
        nearest_vec = None
        min_dist = float('inf')
        for _, row in opps.iterrows():
            dxo = row['posX'] - x0
            dyo = row['posY'] - y0
            dist = np.hypot(dxo, dyo)
            if dist > 0 and dist < min_dist:
                min_dist = dist
                nearest_vec = np.array([dxo, dyo]) / dist
        if nearest_vec is not None:
            ax.arrow(x0, y0, nearest_vec[0]*12, nearest_vec[1]*12,
                     width=0.2, head_width=1.2, color='blue', alpha=0.6,
                     label='Angle to Nearest Opponent' if 'Angle to Nearest Opponent' not in ax.get_legend_handles_labels()[1] else "")

# NOTE: comment translated/cleaned for English-only repository.
    pressure = pass_row.get('pressure_level_num', 0)
    color = {0: 'grey', 1: '#a0c4ff', 2: 'red'}.get(pressure, 'gray')
    circle = Circle((x0, y0), radius=6, color=color, alpha=0.15, label=f'Pressure Level {pressure}')
    ax.add_patch(circle)

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
    plt.close()

# NOTE: comment translated/cleaned for English-only repository.
label_rows = []
count = 0

for idx, pass_row in df_selected.iterrows():
    ts = pass_row['timestamp']
    if pd.isna(ts):
        error_counter["timestamp NaN"] += 1
        continue
    try:
        pass_row_out, df_players = extract_frame_players_and_roles(df_coordinates, pass_row)

        img_name = f"pass_{idx:04d}.png"
        img_path = os.path.join(save_dir, img_name)
        draw_pass_frame_as_image(df_players, pass_row_out, img_path)

        label_rows.append({
    'filename': img_name,
    'is_valuable': int(pass_row['is_valuable']),
    'passedPlayerId': pass_row['passedPlayerId'],          #     
    'window_score': pass_row['window_score'],              #     
    'timestamp': pass_row['timestamp'],                    #            /    
})
        count += 1

    except Exception as e:
        error_counter[str(e)] += 1
        continue

# NOTE: comment translated/cleaned for English-only repository.
df_labels = pd.DataFrame(label_rows)
csv_path = os.path.join(save_dir, "pass_labels_valuable.csv")
df_labels.to_csv(csv_path, index=False)

print(f"Images generated: {count}")
print(f"Label CSV saved to: {csv_path}")
print(df_labels['is_valuable'].value_counts())
print("Error Summary:")
for err, cnt in error_counter.most_common():
    print(f"{err}: {cnt}")





In [ ]:
import os
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tqdm import tqdm

# NOTE: comment translated/cleaned for English-only repository.
img_dir = r"C:\Users\Adolph\Desktop\graduation thesis\predict_image"
model_path = r"C:\Users\Adolph\graduation thesis\resnet50_focal_finetuned_final.h5"
csv_path = os.path.join(img_dir, "pass_labels_valuable.csv")

# NOTE: comment translated/cleaned for English-only repository.
model = load_model(model_path,compile=False)

# NOTE: comment translated/cleaned for English-only repository.
df = pd.read_csv(csv_path)

# NOTE: comment translated/cleaned for English-only repository.
preds = []
for fname in tqdm(df['filename']):
    img_path = os.path.join(img_dir, fname)
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0) / 255.0
    prob = model.predict(x)[0][0]
    preds.append(prob)

df['cnn_score'] = preds
df['cnn_pred'] = (df['cnn_score'] >= 0.5).astype(int)

# NOTE: comment translated/cleaned for English-only repository.
df.to_csv(os.path.join(img_dir, "cnn_predictions.csv"), index=False)
print("  cnn_predictions.csv saved.")
